# M1-INF4 — Python Workshop for Data Analysis
## Session 5 — Data Integration Case Study

### A retail decision supported by reliable data
You are a small analytics team working for **Campus Tech**, a fictional retailer of computer accessories and equipment. The manager wants to compare stores before deciding where to investigate further.

> “The store with the highest total sales is our best-performing store. We should copy its approach everywhere.”

**Your mission:** integrate three sources, assess this claim and deliver an evidence-based recommendation.

This is a synthesis project. Use the pandas and Matplotlib techniques from Sessions 1–4. No new library or advanced modelling is required.

**All data are synthetic. Some records deliberately contain quality issues for training.** Findings describe this supplied sample, not a real company.

### What the data represent
The intended scope is **1–28 April 2025 inclusive**. This is an extract of completed orders, **not a complete record of all activity**. Coverage can differ between stores. A day absent from the extract is not evidence of zero sales or store closure.

| File | One row represents | Expected key | Purpose |
|---|---|---|---|
| `case_sales.csv` | One completed order for one product, possibly several units | `order_id` | Recorded quantity and unit selling price |
| `case_stores.csv` | One store | `store_id` | Store name, city and format |
| `case_products.csv` | One product | `product_id` | Product name and category |

Each order concerns **one product only**, so one valid order row is one order, not a line in a multi-product basket. The price is the final selling price per unit in EUR, after discounts and excluding VAT. Returns and cancellations are outside this extract.

Revenue per order = quantity × unit selling price. **Revenue is not profit**: costs, staffing, opening hours and customer traffic are unavailable.

The reference files should have unique keys, but this must be checked. Repeated store or product IDs in the sales file are normal.

### Data dictionary
| Field | Meaning and rule |
|---|---|
| `order_id` | Order identifier, required and unique after removing exact repeated records |
| `date` | Order date, expected format `YYYY-MM-DD`, within the study window |
| `store_id`, `product_id` | Reference keys; trim spaces and normalise case before joining |
| `quantity` | Positive whole number of units |
| `unit_price_eur` | Positive numeric unit price; a decimal comma may occur |
| `store_name`, `city`, `store_format` | Store attributes; “Small”/“Large” describes format, not observed floor area |
| `product_name`, `category` | Product attributes |

**Quality policy:** keep raw objects unchanged. Remove exact duplicate rows with an audit count. Convert invalid dates/numbers to missing values. Keep invalid or unmatched records in review tables, without inventing replacements. Do not remove a large but valid order simply because it is unusual. If duplicate reference keys disagree, investigate rather than arbitrarily keeping the first row.

### Files and group deliverable
Place this notebook in `notebooks/` and the three CSV files in the neighbouring `data/` folder. You may also put the CSV files beside the notebook.

Submit one completed notebook per group, the exported summary tables and three labelled figures. Record group members below. Divide the work, then review the entire analysis together: **every member must be able to explain the data unit, cleaning choices, joins and conclusions**.

**Group G12**, members NAME 1, NAME 2, NAME 3 and NAME 4 (to complete). Individual contributions are listed in the repository README.

The sections below form one continuous analysis. Each checkpoint is a review of your own results, not a separate dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [Path('../data'), Path('data'), Path('.')]
DATA_DIR = next((p for p in candidates if all((p / f).exists() for f in
    ['case_sales.csv', 'case_stores.csv', 'case_products.csv'])), None)
if DATA_DIR is None:
    raise FileNotFoundError('Place the three case_*.csv files in data/ or beside this notebook.')
OUT_DIR = Path('case_study_outputs')
OUT_DIR.mkdir(exist_ok=True)

sales_raw = pd.read_csv(DATA_DIR / 'case_sales.csv', dtype='string')
stores_raw = pd.read_csv(DATA_DIR / 'case_stores.csv', dtype='string')
products_raw = pd.read_csv(DATA_DIR / 'case_products.csv', dtype='string')
for name, frame in [('Sales', sales_raw), ('Stores', stores_raw), ('Products', products_raw)]:
    print(name, frame.shape)
    display(frame.head())

### Réglages de l'analyse
Cette cellule fixe la période étudiée, calcule une empreinte SHA-256 de chaque fichier brut (revérifiée en section 7 pour prouver que les données brutes n'ont pas été modifiées), règle l'arrondi d'affichage (présentation seulement) et définit un style commun à toutes les figures.

In [ ]:
# Réglages de l'analyse, définis une seule fois puis réutilisés dans toutes les sections
import hashlib
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

# Période étudiée, du 1er au 28 avril 2025 inclus
STUDY_START = pd.Timestamp('2025-04-01')
STUDY_END = pd.Timestamp('2025-04-28')
# Calendrier complet des 28 jours, utilisé pour le tableau journalier et la figure 3
CALENDAR = pd.date_range(STUDY_START, STUDY_END, freq='D')

# Empreinte (SHA-256) de chaque fichier brut et copie des tableaux bruts.
# On les revérifie en section 7 pour prouver que les données brutes n'ont jamais été modifiées.
RAW_FILES = ['case_sales.csv', 'case_stores.csv', 'case_products.csv']
RAW_FINGERPRINTS = {f: hashlib.sha256((DATA_DIR / f).read_bytes()).hexdigest() for f in RAW_FILES}
RAW_SNAPSHOTS = {'sales': sales_raw.copy(deep=True),
                 'stores': stores_raw.copy(deep=True),
                 'products': products_raw.copy(deep=True)}

# Affichage des tableaux. On arrondit seulement à l'affichage, les calculs gardent toute la précision.
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Couleurs communes à toutes les figures (texte sombre et lisible, grille discrète)
INK, INK_2, INK_MUTED = '#0b0b0b', '#52514e', '#898781'
BAR_COLOR, CONTEXT_COLOR = '#2a78d6', '#a9a8a2'
# Une couleur fixe par magasin, choisie pour rester distinguable par les daltoniens.
# Un magasin garde la même couleur dans toutes les figures.
SERIES_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
# Style commun des graphiques (fond blanc, grille fine, pas de cadre en haut ni à droite)
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.titlecolor': INK, 'axes.labelcolor': INK_2, 'text.color': INK,
    'xtick.color': INK_2, 'ytick.color': INK_2,
    'axes.edgecolor': '#c3c2b7', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': False, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.8, 'grid.linestyle': '-',
    'legend.frameon': False,
})

def eur(x, decimals=0):
    """Écrit un montant en euros pour les étiquettes, par exemple 12345.6 devient €12,346."""
    return f'€{x:,.{decimals}f}'

def eur_axis(max_value):
    """Format des graduations en euros, €12k pour les grands montants et €450 sinon."""
    if max_value >= 5000:
        return FuncFormatter(lambda v, _: '€0' if v == 0 else f'€{v / 1000:,.0f}k')
    return FuncFormatter(lambda v, _: eur(v))

def add_titles(fig, title, subtitle, top_inches=0.95):
    """Ajoute en haut de la figure le titre (la question posée) et le sous-titre (la population étudiée)."""
    h = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.10 / h, ha='left', va='top', fontsize=12.5, fontweight='bold')
    fig.text(0.01, 1 - 0.40 / h, subtitle, ha='left', va='top', color=INK_2, fontsize=9)
    fig.subplots_adjust(top=1 - top_inches / h)

print('Data folder used', DATA_DIR)
print('Study window from', STUDY_START.date(), 'to', STUDY_END.date(), f'({len(CALENDAR)} days, inclusive)')
print('Outputs written to', OUT_DIR)

## 1. Understand and audit the sources
1. Report row and column counts, missing values and exact duplicate counts for each source.
2. Check the expected keys. Explain why repeated `store_id` values are acceptable in sales but potentially dangerous in stores.
3. Write the observation unit and the proposed join keys before joining anything.

**Checkpoint:** your audit should distinguish missing data, repeated records and normal repeated foreign keys.

In [ ]:
# Section 1. Audit des trois fichiers (on lit les tableaux bruts sans jamais les modifier)
sources = {'sales': sales_raw, 'stores': stores_raw, 'products': products_raw}
# Clé attendue de chaque fichier, c'est l'identifiant qui devrait être unique
primary_keys = {'sales': 'order_id', 'stores': 'store_id', 'products': 'product_id'}

def is_blank(s):
    """Renvoie True quand une valeur est manquante ou ne contient que des espaces."""
    return s.fillna('').str.strip().eq('').astype(bool)

def normalise_id(s):
    """Règle du dictionnaire de données pour les identifiants.
    On enlève les espaces au début et à la fin, on met en majuscules,
    et un identifiant vide devient une valeur manquante."""
    return s.str.strip().str.upper().replace('', pd.NA)

# 1.1 Vue d'ensemble de chaque fichier (taille, valeurs manquantes, doublons exacts, clé)
rows = []
for name, frame in sources.items():
    key = primary_keys[name]
    # On regarde si la clé se répète encore une fois les doublons exacts retirés
    key_after_dedup = normalise_id(frame.drop_duplicates()[key])
    repeated = key_after_dedup[key_after_dedup.notna() & key_after_dedup.duplicated(keep=False)]
    rows.append({
        'source': name,
        'rows': len(frame),
        'columns': frame.shape[1],
        'missing cells (NA)': int(frame.isna().sum().sum()),
        'blank-text cells': int(sum((is_blank(frame[c]) & frame[c].notna()).sum() for c in frame.columns)),
        'rows with at least 1 missing value': int(frame.isna().any(axis=1).sum()),
        'exact duplicate rows (extra copies)': int(frame.duplicated().sum()),
        'expected key': key,
        'rows with missing key': int(normalise_id(frame[key]).isna().sum()),
        'keys still repeated after removing exact duplicates': int(repeated.nunique()),
    })
audit_overview = pd.DataFrame(rows).set_index('source')
print('1.1 Overview of the three sources')
display(audit_overview.T)

# 1.2 Valeurs manquantes colonne par colonne
# (NA = case vide dans le fichier, blank text = texte qui ne contient que des espaces)
rows = []
for name, frame in sources.items():
    for col in frame.columns:
        rows.append({'source': name, 'column': col,
                     'missing (NA)': int(frame[col].isna().sum()),
                     'blank text': int((is_blank(frame[col]) & frame[col].notna()).sum()),
                     'distinct values': int(frame[col].nunique())})
missing_by_column = pd.DataFrame(rows)
print('1.2 Missing values by column')
display(missing_by_column)

# 1.3 Doublons exacts. On affiche toutes les copies pour pouvoir les vérifier à l'oeil.
for name, frame in sources.items():
    copies = frame[frame.duplicated(keep=False)]
    if len(copies):
        print(f'1.3 {name}, {frame.duplicated().sum()} extra copies of identical rows')
        display(copies.sort_values(list(frame.columns)))

# 1.4 Dans stores et products, la clé doit être unique.
# Sinon une jointure dupliquerait les commandes (une commande trouverait deux magasins).
for name in ['stores', 'products']:
    key = primary_keys[name]
    dedup = sources[name].drop_duplicates()
    k = normalise_id(dedup[key])
    clash = dedup[k.notna() & k.duplicated(keep=False)]
    print(f'1.4 {name}, {k.nunique()} distinct {key} after trimming and upper-casing, '
          f'{len(clash)} non-identical rows share a {key}')
    if len(clash):
        display(clash.assign(normalised_key=k[clash.index]).sort_values('normalised_key'))

# 1.5 Dans sales, store_id et product_id se répètent normalement
# (un magasin ou un produit apparaît dans plusieurs commandes, ce sont des clés étrangères)
fk = pd.DataFrame({
    'distinct raw values': [sales_raw[c].nunique() for c in ['store_id', 'product_id']],
    'distinct after trim + upper-case': [normalise_id(sales_raw[c]).nunique() for c in ['store_id', 'product_id']],
    'orders per key (median)': [normalise_id(sales_raw[c]).value_counts().median() for c in ['store_id', 'product_id']],
    'missing': [int(normalise_id(sales_raw[c]).isna().sum()) for c in ['store_id', 'product_id']],
}, index=['store_id', 'product_id'])
print('1.5 Foreign keys in sales (repetition is normal here)')
display(fk)

# 1.6 Formats à convertir ou à vérifier en section 2
# (dates hors format AAAA-MM-JJ, quantités non entières, virgules décimales, identifiants mal écrits)
txt = {c: sales_raw[c].fillna('').str.strip() for c in ['date', 'quantity', 'unit_price_eur', 'store_id', 'product_id']}
format_issues = pd.Series({
    'date missing or not written YYYY-MM-DD': int((~txt['date'].str.fullmatch(r'\d{4}-\d{2}-\d{2}')).sum()),
    'quantity missing or not written as digits only': int((~txt['quantity'].str.fullmatch(r'\d+')).sum()),
    'unit_price_eur written with a decimal comma': int(txt['unit_price_eur'].str.contains(',', regex=False).sum()),
    'unit_price_eur missing or not a plain number': int((~txt['unit_price_eur'].str.replace(',', '.', regex=False)
                                                         .str.fullmatch(r'-?\d+(\.\d+)?')).sum()),
    'store_id with spaces or lower case': int((sales_raw['store_id'].notna()
                                               & sales_raw['store_id'].ne(normalise_id(sales_raw['store_id'])).fillna(True)).sum()),
    'product_id with spaces or lower case': int((sales_raw['product_id'].notna()
                                                 & sales_raw['product_id'].ne(normalise_id(sales_raw['product_id'])).fillna(True)).sum()),
}, name='raw sales rows')
print('1.6 Formats to convert or check (raw sales rows)')
display(format_issues.to_frame())

**Your interpretation**

What does one row mean in each source? Which columns connect the sources?

*Write your answer here.*

## 2. Prepare reliable tables
Create working copies. Apply the quality policy above.

1. Count and remove **exact** duplicate records in all three tables.
2. Normalise identifiers, parse dates and convert the two numeric fields. Handle decimal commas.
3. Flag missing IDs, failed conversions, non-positive prices, quantities that are not positive integers, and dates outside the study window.
4. Produce `sales_valid` and `sales_review`. Keep the rejected records and the reason flags.
5. Validate unique, non-missing reference keys and unique valid order IDs.

**Checkpoint:** input rows = exact duplicates removed + invalid rows retained for review + valid rows. Multiple reason flags can describe the same rejected row, so do not sum flags as if they were distinct rows.

In [ ]:
# Section 2. Nettoyage et validation, sans jamais modifier les tableaux bruts
SALES_COLS = list(sales_raw.columns)

# 2.1 Doublons exacts dans les ventes
# On travaille sur une copie. On ajoute le numéro de ligne du CSV (l'en-tête est la ligne 1)
# pour pouvoir retrouver chaque ligne rejetée dans le fichier d'origine.
sales_work = sales_raw.copy()
sales_work.insert(0, 'source_line', np.arange(2, len(sales_work) + 2))
# Une ligne est un doublon exact si toutes ses colonnes d'origine sont identiques à une ligne précédente.
# On garde la première copie et on compte les autres.
exact_dup = sales_work.duplicated(subset=SALES_COLS, keep='first')
sales_exact_duplicates = sales_work[exact_dup]
sales_dedup = sales_work[~exact_dup].reset_index(drop=True)
N_EXACT_DUP = int(exact_dup.sum())
print(f'2.1 Sales, {N_EXACT_DUP} exact duplicate rows removed '
      f'({len(sales_work)} rows before, {len(sales_dedup)} after)')

# 2.2 Normalisation des identifiants et conversion des dates et des nombres
def to_number(s):
    """Convertit un texte en nombre. La virgule décimale (12,5) est acceptée.
    Tout ce qui reste illisible devient une valeur manquante (NaN), on ne devine rien."""
    txt = s.fillna('').str.strip()
    # Une virgule sans point est une virgule décimale, on la remplace par un point
    comma_decimal = (txt.str.contains(',', regex=False) & ~txt.str.contains('.', regex=False)).astype(bool)
    txt = txt.mask(comma_decimal, txt.str.replace(',', '.', regex=False))
    num = pd.to_numeric(txt.astype(object), errors='coerce').astype(float)
    return num.where(np.isfinite(num))

sales_clean = pd.DataFrame({
    'source_line': sales_dedup['source_line'],
    'order_id': normalise_id(sales_dedup['order_id']),
    # Format strict AAAA-MM-JJ. Une date impossible ou mal écrite devient NaT (date manquante).
    'date': pd.to_datetime(sales_dedup['date'].fillna('').str.strip().astype(object),
                           format='%Y-%m-%d', errors='coerce'),
    'store_id': normalise_id(sales_dedup['store_id']),
    'product_id': normalise_id(sales_dedup['product_id']),
    'quantity': to_number(sales_dedup['quantity']),
    'unit_price_eur': to_number(sales_dedup['unit_price_eur']),
})

# 2.3 Un indicateur (True ou False) par motif de rejet.
# On compare avec le texte brut pour distinguer une valeur absente d'une valeur présente mais illisible.
raw = sales_dedup
q, p, d = sales_clean['quantity'], sales_clean['unit_price_eur'], sales_clean['date']
flags = pd.DataFrame({
    'missing_order_id': sales_clean['order_id'].isna(),
    'missing_store_id': sales_clean['store_id'].isna(),
    'missing_product_id': sales_clean['product_id'].isna(),
    'missing_date': is_blank(raw['date']),
    'invalid_date': ~is_blank(raw['date']) & d.isna(),
    'date_outside_window': d.notna() & ~d.between(STUDY_START, STUDY_END),
    'missing_quantity': is_blank(raw['quantity']),
    'invalid_quantity': ~is_blank(raw['quantity']) & q.isna(),
    'quantity_not_positive_integer': q.notna() & ((q <= 0) | (q % 1 != 0)),
    'missing_price': is_blank(raw['unit_price_eur']),
    'invalid_price': ~is_blank(raw['unit_price_eur']) & p.isna(),
    'price_not_positive': p.notna() & (p <= 0),
})
# Même commande écrite deux fois avec seulement une différence de forme (12,50 et 12.50, s01 et S01).
# Après normalisation les deux lignes sont identiques. On garde la première, les suivantes vont en revue.
NORM_COLS = ['order_id', 'date', 'store_id', 'product_id', 'quantity', 'unit_price_eur']
flags['duplicate_after_normalisation'] = (sales_clean['order_id'].notna()
                                          & sales_clean.duplicated(subset=NORM_COLS, keep='first'))
# Sinon un order_id doit désigner une seule commande. S'il apparaît sur des lignes qui se contredisent,
# on ne peut pas savoir laquelle est juste, donc toutes les copies vont en revue.
oid = sales_clean['order_id'].where(~flags['duplicate_after_normalisation'])
flags['repeated_order_id'] = oid.notna() & oid.duplicated(keep=False)
FLAG_COLS = list(flags.columns)

# Nombre de problèmes par ligne et liste lisible des motifs. Une ligne est valide si elle n'a aucun problème.
n_issues = flags.sum(axis=1)
review_reasons = flags.apply(lambda row: ', '.join(row.index[row.to_numpy()]), axis=1)
is_valid = n_issues.eq(0)

# 2.4 Séparation en lignes valides (sales_valid) et lignes à revoir (sales_review)
sales_valid = (sales_clean.loc[is_valid, ['order_id', 'date', 'store_id', 'product_id', 'quantity', 'unit_price_eur']]
               .reset_index(drop=True).copy())
# Toutes les quantités restantes sont des nombres entiers positifs, on les stocke en entiers
sales_valid['quantity'] = sales_valid['quantity'].astype(int)

# sales_review garde le texte brut tel qu'il était dans le fichier, plus les indicateurs et les motifs
sales_review = pd.concat([sales_dedup.loc[~is_valid],
                          flags.loc[~is_valid],
                          n_issues[~is_valid].rename('n_issues'),
                          review_reasons[~is_valid].rename('review_reasons')], axis=1).reset_index(drop=True)

# Une même ligne peut avoir plusieurs motifs, donc on ne doit pas additionner les compteurs de motifs
flag_summary = pd.DataFrame({'rows flagged': flags.sum(),
                             '% of deduplicated rows': 100 * flags.mean()})
print('2.3 Reason flags (one row can carry several flags, so these counts are not additive)')
display(flag_summary)
print('Rejected rows by number of flags')
display(n_issues[~is_valid].value_counts().sort_index().rename_axis('number of flags').to_frame('rejected rows'))
print(f'2.4 sales_review contains {len(sales_review)} rows kept for review (first 15 shown)')
display(sales_review[['source_line'] + SALES_COLS + ['review_reasons']].head(15))

# Réparations de forme faites sur des lignes gardées comme valides (aucune valeur inventée)
repairs = pd.Series({
    'unit_price_eur, decimal comma read as decimal point':
        int((raw['unit_price_eur'].fillna('').str.contains(',', regex=False) & is_valid).sum()),
    'store_id, spaces trimmed or upper-cased':
        int((raw['store_id'].fillna('').ne(sales_clean['store_id'].fillna('')) & is_valid).sum()),
    'product_id, spaces trimmed or upper-cased':
        int((raw['product_id'].fillna('').ne(sales_clean['product_id'].fillna('')) & is_valid).sum()),
    'date not zero-padded but unambiguous (e.g. 2025-4-5)':
        int((~raw['date'].fillna('').str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}') & is_valid).sum()),
}, name='valid rows repaired')
print('Formatting repairs on valid rows')
display(repairs.to_frame())

# Les grosses commandes sont gardées. Inhabituel ne veut pas dire faux.
order_value = sales_valid['quantity'] * sales_valid['unit_price_eur']
print('Largest valid orders (kept, nothing indicates an error)')
display(sales_valid.assign(order_value_eur=order_value).nlargest(5, 'order_value_eur'))

# 2.5 Nettoyage des tables de référence (stores et products)
def harmonise_labels(s):
    """Uniformise l'écriture d'une étiquette. small, SMALL et Small deviennent la même étiquette,
    on garde l'écriture la plus fréquente (en cas d'égalité, l'ordre alphabétique)."""
    s = s.str.strip().str.replace(r'\s+', ' ', regex=True).replace('', pd.NA)
    lower = s.str.lower()
    spelling = s.dropna().groupby(lower.dropna()).agg(lambda x: x.value_counts().sort_index().idxmax())
    return lower.map(spelling).astype('string')

# Ce qui arrive à une clé répétée avec des informations différentes
QUARANTINED = 'key quarantined, its orders go to integration_review'
KEPT_UNRESOLVED = 'kept once, disputed attribute set to missing (not guessed)'

def clean_reference(raw_table, key, identity_cols, label_cols, text_cols, note_col):
    """Nettoie une table de référence en 4 étapes.
    1. On retire les doublons exacts.
    2. On normalise la clé, on nettoie les textes et on uniformise les étiquettes.
    3. Les lignes devenues identiques après ce nettoyage sont fusionnées.
    4. Une clé encore répétée avec des informations différentes est examinée, jamais réglée en gardant la première ligne.
       Si les lignes diffèrent sur l'identité (nom, ville), la clé pourrait désigner deux entités différentes,
       elle est mise en quarantaine et ses commandes ne peuvent pas être attribuées.
       Si elles diffèrent seulement sur une étiquette descriptive (format, catégorie), c'est la même entité,
       on la garde une seule fois et l'étiquette contestée devient manquante."""
    work = raw_table.drop_duplicates().copy()
    audit = {'rows in': len(raw_table), 'exact duplicates removed': len(raw_table) - len(work)}
    work[key] = normalise_id(work[key])
    for c in text_cols:
        work[c] = work[c].str.strip().str.replace(r'\s+', ' ', regex=True).replace('', pd.NA)
    relabelled = 0
    for c in label_cols:
        new = harmonise_labels(work[c])
        relabelled += int((work[c].fillna('').str.strip() != new.fillna('')).sum())
        work[c] = new
    audit['label spellings harmonised'] = relabelled
    same_after_tidy = work.duplicated()
    audit['rows identical after tidying (collapsed)'] = int(same_after_tidy.sum())
    work = work[~same_after_tidy].reset_index(drop=True)

    # Clés manquantes et clés encore répétées, avec la liste des colonnes qui se contredisent
    missing_key = work[key].isna()
    repeated = work[key].notna() & work[key].duplicated(keep=False)
    disputed = {k: [c for c in work.columns if c != key and g[c].nunique(dropna=False) > 1]
                for k, g in work[repeated].groupby(key)}
    identity_conflict = [k for k, cols in disputed.items() if set(cols) & set(identity_cols)]
    label_conflict = [k for k in disputed if k not in identity_conflict]

    # Table de revue des références, avec la décision prise pour chaque ligne
    review = work[missing_key | repeated].copy()
    review['attributes_in_conflict'] = review[key].map(lambda k: ', '.join(disputed.get(k, []))).fillna('')
    review['action'] = np.where(review[key].isna(), 'excluded, no key to join on',
                                np.where(review[key].isin(identity_conflict), QUARANTINED, KEPT_UNRESOLVED))

    # Table propre, une ligne par clé. Pour un conflit d'étiquette, la valeur contestée devient manquante.
    clean = work[~missing_key & ~repeated].copy()
    clean[note_col] = ''
    resolved = []
    for k in label_conflict:
        grp = work[work[key].eq(k)]
        row = {c: (pd.NA if c in disputed[k] else grp[c].iloc[0]) for c in work.columns}
        row[note_col] = ', '.join(disputed[k])
        resolved.append(row)
    if resolved:
        clean = pd.concat([clean, pd.DataFrame(resolved)], ignore_index=True)
    clean = clean.astype({c: 'string' for c in work.columns}).sort_values(key).reset_index(drop=True)
    audit.update({'rows with missing key': int(missing_key.sum()),
                  'keys quarantined (identity conflict)': len(identity_conflict),
                  'keys kept with a disputed label set to missing': len(label_conflict),
                  'rows in clean table': len(clean)})
    return clean, review.reset_index(drop=True), audit

# Nom et ville identifient un magasin, le format est une simple étiquette.
# Le nom identifie un produit, la catégorie est une simple étiquette.
stores_clean, stores_review, stores_audit = clean_reference(
    stores_raw, 'store_id', identity_cols=['store_name', 'city'], label_cols=['city', 'store_format'],
    text_cols=['store_name'], note_col='store_unresolved')
products_clean, products_review, products_audit = clean_reference(
    products_raw, 'product_id', identity_cols=['product_name'], label_cols=['category'],
    text_cols=['product_name'], note_col='product_unresolved')
reference_audit = pd.DataFrame({'stores': stores_audit, 'products': products_audit})
print('2.5 Reference tables')
display(reference_audit)

for name, review in [('stores', stores_review), ('products', products_review)]:
    if len(review):
        print(f'{name}, rows that needed investigation')
        display(review)

# 2.6 Vérifications finales. Le programme s'arrête avec une erreur si une seule est fausse.
# Clés uniques et non manquantes dans les références et dans les commandes valides
assert stores_clean['store_id'].notna().all() and stores_clean['store_id'].is_unique
assert products_clean['product_id'].notna().all() and products_clean['product_id'].is_unique
assert sales_valid['order_id'].notna().all() and sales_valid['order_id'].is_unique
# Lignes en entrée = doublons exacts retirés + lignes en revue + lignes valides
assert len(sales_raw) == N_EXACT_DUP + len(sales_review) + len(sales_valid)

cleaning_check = pd.Series({'input rows (case_sales.csv)': len(sales_raw),
                            'exact duplicates removed': N_EXACT_DUP,
                            'invalid or repeated rows kept in sales_review': len(sales_review),
                            'valid rows (sales_valid)': len(sales_valid)}, name='rows')
print('2.6 Checkpoint OK, input rows = exact duplicates removed + rows kept for review + valid rows')
display(cleaning_check.to_frame())
print('Keys checked, store_id and product_id are unique in the references and order_id is unique in sales_valid')

**Your interpretation**

Which records did you exclude, repair or keep? Why?

*Write your answer here.*

## 3. Integrate and reconcile
1. Join `sales_valid` to stores with a **left join**, `validate="many_to_one"` and an indicator. Inspect unmatched keys.
2. Join that result to products with the same controls. Use a different indicator name, or rename/remove the first indicator before the second join.
3. Check that neither join changes the row count.
4. Keep orders missing either reference in `integration_review`. Create `analysis` using orders matched to **both** references.
5. Present a reconciliation table from raw input to the final analytical population.

**Checkpoint:** never interpret an unknown product as zero revenue, and never allow a duplicated reference key to multiply orders. State how many orders the analysis excludes.

In [ ]:
# Section 3. Jointures et contrôle des effectifs
# Clés mises en quarantaine en section 2 (conflit d'identité dans les fichiers de référence)
quarantined_stores = stores_review.loc[stores_review['action'].eq(QUARANTINED), 'store_id']
quarantined_products = products_review.loc[products_review['action'].eq(QUARANTINED), 'product_id']

# 3.1 Jointure à gauche des commandes valides avec les magasins.
# how='left' garde toutes les commandes, même celles dont le magasin est inconnu.
# validate='many_to_one' vérifie qu'un store_id n'apparaît qu'une fois dans stores_clean
# (plusieurs commandes pour un magasin, jamais l'inverse), sinon pandas s'arrête avec une erreur.
# indicator='store_match' ajoute une colonne qui dit si le magasin a été trouvé (both) ou non (left_only).
orders_stores = sales_valid.merge(stores_clean, on='store_id', how='left',
                                  validate='many_to_one', indicator='store_match')
# Contrôle des effectifs, la jointure ne doit ni ajouter ni perdre de lignes
assert len(orders_stores) == len(sales_valid), 'the store join changed the number of rows'
print(f'3.1 Store join, {len(sales_valid)} rows before and {len(orders_stores)} rows after, no row was multiplied')
print(orders_stores['store_match'].value_counts().to_string(), '\n')

def unmatched_keys(frame, key, indicator, quarantined):
    """Liste des clés non trouvées, avec leur nombre de commandes et la raison de l'échec."""
    out = (frame.loc[frame[indicator].eq('left_only'), key]
           .value_counts().rename_axis(key).reset_index(name='orders'))
    out['reason'] = np.where(out[key].isin(quarantined), 'quarantined, conflicting rows in reference file',
                             'absent from reference file')
    return out

display(unmatched_keys(orders_stores, 'store_id', 'store_match', quarantined_stores))

# 3.2 Deuxième jointure à gauche, avec les produits.
# On utilise un autre nom d'indicateur (product_match) pour ne pas écraser le premier.
orders_full = orders_stores.merge(products_clean, on='product_id', how='left',
                                  validate='many_to_one', indicator='product_match')
assert len(orders_full) == len(orders_stores), 'the product join changed the number of rows'
print(f'3.2 Product join, {len(orders_stores)} rows before and {len(orders_full)} rows after, no row was multiplied')
print(orders_full['product_match'].value_counts().to_string(), '\n')
display(unmatched_keys(orders_full, 'product_id', 'product_match', quarantined_products))

# 3.3 Séparation entre analysis (magasin ET produit trouvés) et integration_review (le reste)
store_ok = orders_full['store_match'].eq('both').to_numpy()
product_ok = orders_full['product_match'].eq('both').to_numpy()
# On écrit en clair pourquoi chaque commande exclue n'a pas pu être rattachée
store_issue = np.where(store_ok, '', np.where(orders_full['store_id'].isin(quarantined_stores),
                       'store_id quarantined (conflicting reference rows)', 'store_id not in case_stores.csv'))
product_issue = np.where(product_ok, '', np.where(orders_full['product_id'].isin(quarantined_products),
                         'product_id quarantined (conflicting reference rows)', 'product_id not in case_products.csv'))

orders_full['store_matched'] = store_ok
orders_full['product_matched'] = product_ok
orders_full['integration_issue'] = [' and '.join(t for t in pair if t) for pair in zip(store_issue, product_issue)]

integration_review = (orders_full.loc[~(store_ok & product_ok)]
                      .drop(columns=['store_match', 'product_match']).reset_index(drop=True).copy())
# Valeur des commandes exclues. Elle est connue mais n'est ajoutée à aucun magasin ni produit,
# et un produit inconnu n'est jamais compté comme 0 euro.
integration_review['order_value_eur'] = integration_review['quantity'] * integration_review['unit_price_eur']

analysis = (orders_full.loc[store_ok & product_ok]
            .drop(columns=['store_match', 'product_match', 'store_matched', 'product_matched', 'integration_issue'])
            .reset_index(drop=True).copy())

# Chaque commande valide est soit dans analysis soit dans integration_review, jamais les deux
assert len(analysis) + len(integration_review) == len(sales_valid)
assert analysis['order_id'].is_unique
print(f'3.3 analysis has {len(analysis)} orders matched to both references, '
      f'integration_review has {len(integration_review)} orders')
display(integration_review[['order_id', 'date', 'store_id', 'product_id', 'quantity', 'unit_price_eur',
                            'order_value_eur', 'integration_issue']])

# 3.4 Tableau de réconciliation, du fichier brut jusqu'à la population analysée.
# Les nombres négatifs sont les lignes retirées à chaque étape.
value = lambda df: float((df['quantity'] * df['unit_price_eur']).sum())
only_store_missing = integration_review[~integration_review['store_matched'] & integration_review['product_matched']]
only_product_missing = integration_review[integration_review['store_matched'] & ~integration_review['product_matched']]
both_missing = integration_review[~integration_review['store_matched'] & ~integration_review['product_matched']]

reconciliation = pd.DataFrame([
    ('Raw rows in case_sales.csv', len(sales_raw), np.nan),
    ('Exact duplicate rows removed', -N_EXACT_DUP, np.nan),
    ('Rows after removing exact duplicates', len(sales_dedup), np.nan),
    ('Invalid or repeated rows kept in sales_review', -len(sales_review), np.nan),
    ('Valid orders (sales_valid)', len(sales_valid), value(sales_valid)),
    ('Valid orders with store unknown or quarantined (product matched)', -len(only_store_missing),
     -value(only_store_missing)),
    ('Valid orders with product unknown or quarantined (store matched)', -len(only_product_missing),
     -value(only_product_missing)),
    ('Valid orders with neither reference matched', -len(both_missing), -value(both_missing)),
    ('Orders in the analysis (matched to both references)', len(analysis), value(analysis)),
], columns=['step', 'rows', 'order_value_eur'])

# Le calcul du tableau doit tomber juste à chaque étape
assert len(sales_raw) - N_EXACT_DUP == len(sales_dedup)
assert len(sales_dedup) - len(sales_review) == len(sales_valid)
assert len(sales_valid) - len(integration_review) == len(analysis)
assert np.isclose(value(sales_valid) - integration_review['order_value_eur'].sum(), value(analysis))
print('3.4 Reconciliation from raw input to the analytical population (all checks passed)')
display(reconciliation)

# Nombre de commandes exclues de l'analyse, à citer dans l'interprétation
N_EXCLUDED_ORDERS = len(sales_review) + len(integration_review)
print(f'The analysis keeps {len(analysis)} of {len(sales_dedup)} distinct records '
      f'({100 * len(analysis) / len(sales_dedup):.1f}%). It excludes {N_EXCLUDED_ORDERS} records, '
      f'{len(sales_review)} invalid or repeated and {len(integration_review)} valid but unmatched '
      f'(worth {eur(integration_review["order_value_eur"].sum())}). '
      f'In addition, {N_EXACT_DUP} exact duplicate copies were removed beforehand.')

**Your interpretation**

Why use left joins first, even though the analysis ultimately uses matched orders?

*Write your answer here.*

## 4. Build indicators that answer the question
For each order in `analysis`, calculate `revenue_eur`.

Create `store_summary` with, for each store:
- order count and total units;
- total revenue (EUR);
- mean and median revenue per order (EUR).

Also create a revenue summary by product category and a daily revenue table for each store. Check that the totals reconcile with the order-level revenue.

Compare the store with the highest total revenue to the store with the highest mean revenue per order. Do they coincide? Explain the role of order counts and product mix.

**Important:** `quantity` counts units, not orders. Mean revenue per order = total revenue / number of orders. Round for presentation, not before aggregation. Avoid calling this a profitability or productivity ranking.

In [ ]:
# Section 4. Indicateurs
# 4.1 Chiffre d'affaires de chaque commande = quantité x prix unitaire (on ne l'arrondit pas)
analysis['revenue_eur'] = analysis['quantity'] * analysis['unit_price_eur']
# Étiquette de catégorie pour les regroupements. Une catégorie inconnue est affichée telle quelle,
# jamais supprimée ni devinée.
category_disputed = analysis['product_unresolved'].fillna('').str.contains('category').to_numpy(dtype=bool)
analysis['category_label'] = np.where(category_disputed, '(category disputed in reference)',
                                      analysis['category'].fillna('(category missing in reference)'))
TOTAL_REVENUE = analysis['revenue_eur'].sum()
N_ORDERS = len(analysis)
# Phrase qui décrit la population étudiée, réutilisée dans les sous-titres des figures
POPULATION = (f'{N_ORDERS} valid orders matched to store and product references, '
              f'extract of 1 to 28 April 2025, revenue in EUR excl. VAT')
print('Population analysed is', POPULATION)
print(f'Total revenue in the analysis is {eur(TOTAL_REVENUE, 2)}')

# 4.2 Tableau par magasin (store_summary).
# On regroupe seulement par store_id puis on ajoute le nom, la ville et le format,
# comme ça une information manquante ne peut pas faire disparaître un magasin.
store_kpis = (analysis.groupby('store_id')
              .agg(order_count=('order_id', 'size'),                 # nombre de commandes
                   total_units=('quantity', 'sum'),                  # nombre d'unités vendues
                   total_revenue_eur=('revenue_eur', 'sum'),         # chiffre d'affaires total
                   median_revenue_per_order_eur=('revenue_eur', 'median'),
                   days_with_orders=('date', 'nunique'))             # jours présents dans l'extrait
              .reset_index())
# Moyenne par commande = chiffre d'affaires total / nombre de commandes (et non / nombre d'unités)
store_kpis['mean_revenue_per_order_eur'] = store_kpis['total_revenue_eur'] / store_kpis['order_count']
store_kpis['revenue_share_pct'] = 100 * store_kpis['total_revenue_eur'] / TOTAL_REVENUE
store_kpis['units_per_order'] = store_kpis['total_units'] / store_kpis['order_count']
store_kpis['revenue_per_unit_eur'] = store_kpis['total_revenue_eur'] / store_kpis['total_units']

store_summary = (stores_clean.merge(store_kpis, on='store_id', how='inner', validate='one_to_one')
                 .sort_values('total_revenue_eur', ascending=False).reset_index(drop=True))
# Rang de chaque magasin selon le total et selon la moyenne par commande (1 = le plus élevé)
store_summary['rank_total_revenue'] = store_summary['total_revenue_eur'].rank(ascending=False, method='min').astype(int)
store_summary['rank_mean_order_revenue'] = (store_summary['mean_revenue_per_order_eur']
                                            .rank(ascending=False, method='min').astype(int))
store_summary = store_summary[['store_id', 'store_name', 'city', 'store_format',
                               'order_count', 'total_units', 'total_revenue_eur', 'revenue_share_pct',
                               'mean_revenue_per_order_eur', 'median_revenue_per_order_eur',
                               'units_per_order', 'revenue_per_unit_eur', 'days_with_orders',
                               'rank_total_revenue', 'rank_mean_order_revenue']]
print('\n4.2 store_summary (sorted by total revenue)')
display(store_summary)

# Un magasin de la référence sans commande dans l'extrait n'est pas forcément un magasin sans ventes
absent = sorted(set(stores_clean['store_id']) - set(store_summary['store_id']))
if absent:
    print('Stores in the reference with no order in the analysis (not evidence of zero sales)', absent)

# 4.3 Tableau par catégorie de produit (category_summary)
category_summary = (analysis.groupby('category_label')
                    .agg(order_count=('order_id', 'size'),
                         total_units=('quantity', 'sum'),
                         total_revenue_eur=('revenue_eur', 'sum'),
                         median_revenue_per_order_eur=('revenue_eur', 'median'))
                    .reset_index().rename(columns={'category_label': 'category'}))
category_summary['mean_revenue_per_order_eur'] = category_summary['total_revenue_eur'] / category_summary['order_count']
category_summary['revenue_share_pct'] = 100 * category_summary['total_revenue_eur'] / TOTAL_REVENUE
category_summary = (category_summary.sort_values('total_revenue_eur', ascending=False).reset_index(drop=True)
                    [['category', 'order_count', 'total_units', 'total_revenue_eur', 'revenue_share_pct',
                      'mean_revenue_per_order_eur', 'median_revenue_per_order_eur']])
print('4.3 category_summary')
display(category_summary)

# 4.4 Chiffre d'affaires par jour et par magasin.
# reindex(CALENDAR) met les 28 jours pour tous les magasins. Un jour sans commande dans l'extrait
# reste vide (NaN) et n'est surtout pas remplacé par 0, car l'extrait n'est pas complet.
daily_store_revenue = (analysis.groupby(['date', 'store_id'])
                       .agg(orders=('order_id', 'size'), revenue_eur=('revenue_eur', 'sum'))
                       .reset_index())
daily_revenue = (daily_store_revenue.pivot(index='date', columns='store_id', values='revenue_eur')
                 .reindex(CALENDAR))
daily_revenue.index.name = 'date'
# Couverture, nombre de jours avec des commandes et nombre de jours absents pour chaque magasin
coverage = pd.DataFrame({'days with recorded orders': daily_revenue.notna().sum(),
                         'days absent from the extract': daily_revenue.isna().sum()})
print('4.4 Daily revenue table in EUR (NaN means no record in the extract, not zero)')
display(daily_revenue.round(0))
display(coverage.T)

# 4.5 Les totaux des trois tableaux doivent être égaux au total calculé commande par commande
checks = pd.DataFrame({
    'revenue_eur': [TOTAL_REVENUE, store_summary['total_revenue_eur'].sum(),
                    category_summary['total_revenue_eur'].sum(), np.nansum(daily_revenue.to_numpy())],
    'orders': [N_ORDERS, store_summary['order_count'].sum(), category_summary['order_count'].sum(),
               daily_store_revenue['orders'].sum()],
}, index=['order level (analysis)', 'store_summary', 'category_summary', 'daily table'])
assert np.allclose(checks['revenue_eur'], TOTAL_REVENUE) and (checks['orders'] == N_ORDERS).all()
assert np.allclose(store_summary['order_count'] * store_summary['mean_revenue_per_order_eur'],
                   store_summary['total_revenue_eur'])
print('4.5 All totals reconcile with the order-level revenue (checks passed)')
display(checks)

# 4.6 Le magasin en tête sur le total est-il aussi en tête sur la moyenne et la médiane par commande ?
lead_total = store_summary.loc[store_summary['total_revenue_eur'].idxmax()]
lead_mean = store_summary.loc[store_summary['mean_revenue_per_order_eur'].idxmax()]
lead_median = store_summary.loc[store_summary['median_revenue_per_order_eur'].idxmax()]
for label, row, col in [('total revenue', lead_total, 'total_revenue_eur'),
                        ('mean revenue per order', lead_mean, 'mean_revenue_per_order_eur'),
                        ('median revenue per order', lead_median, 'median_revenue_per_order_eur')]:
    print(f'Highest {label} is {row["store_name"]} ({row["store_id"]}) with {eur(row[col])} '
          f'from {row["order_count"]} orders')
print('Same store leads total and mean revenue?', 'yes' if lead_total['store_id'] == lead_mean['store_id'] else 'no')

# Décomposition qui explique les écarts.
# Total = nombre de commandes x moyenne par commande, et moyenne = unités par commande x prix moyen par unité.
decomposition = store_summary[['store_id', 'store_name', 'order_count', 'mean_revenue_per_order_eur',
                               'total_revenue_eur', 'units_per_order', 'revenue_per_unit_eur', 'days_with_orders']]
print('\nDecomposition, total revenue = number of orders x mean order revenue, '
      'and mean order revenue = units per order x revenue per unit')
display(decomposition)

# 4.7 Mix produit, part de chaque catégorie dans le chiffre d'affaires de chaque magasin.
# Le dénominateur est le total du magasin, donc chaque ligne fait 100 %.
store_order = store_summary['store_id'].tolist()
mix_revenue = analysis.pivot_table(index='store_id', columns='category_label', values='revenue_eur',
                                   aggfunc='sum', fill_value=0).reindex(store_order)
mix_share = 100 * mix_revenue.div(mix_revenue.sum(axis=1), axis=0)
mix_share = mix_share[category_summary['category'].tolist()]
mix_share.columns.name = 'category'
print("4.7 Share of each store's revenue by category in % (each row sums to 100)")
display(mix_share.round(1))

# 4.8 Test de sensibilité. Si on rajoutait les commandes exclues dont le magasin est connu
# (seul le produit manque), le classement des magasins changerait-il ?
at_stake = (integration_review[integration_review['store_matched']]
            .groupby('store_id')['order_value_eur'].sum())
sensitivity = store_summary[['store_id', 'store_name', 'total_revenue_eur', 'rank_total_revenue']].copy()
sensitivity['excluded_value_known_store_eur'] = sensitivity['store_id'].map(at_stake).fillna(0.0)
sensitivity['upper_bound_total_eur'] = sensitivity['total_revenue_eur'] + sensitivity['excluded_value_known_store_eur']
sensitivity['rank_if_added'] = sensitivity['upper_bound_total_eur'].rank(ascending=False, method='min').astype(int)
unattributable = integration_review.loc[~integration_review['store_matched'], 'order_value_eur'].sum()
print('4.8 Sensitivity check (excluded orders whose store is known but product is not)')
display(sensitivity)
print(f'Valid orders that cannot be attributed to any known store are worth {eur(unattributable)}')

**Your interpretation**

Does the same store lead for total and mean revenue? What can explain a difference?

*Write your answer here.*

## 5. Communicate with three complementary figures
Before coding each figure, write **the question**, **the observation unit** and **the aggregation**.

1. A bar chart comparing total revenue by store, with order counts available alongside it.
2. A boxplot of order revenue by store, showing centre, spread and atypical values. Display group sizes. Do not remove atypical values without evidence of error.
3. A line chart of daily revenue by store over the study window. Use chronological dates, one labelled colour per store and a common calendar. **Leave missing store-days as missing**, not zero, because this is an extract.

Use clear titles, units, readable labels and legends when needed. Add a two-sentence interpretation and a limitation for each figure. Explain why the three figures answer different questions.

In [ ]:
# Figure 1. Chiffre d'affaires total par magasin, avec le nombre de commandes à côté
# Question posée. Quels magasins font le plus de chiffre d'affaires dans l'extrait, et avec combien de commandes ?
# Unité d'observation. Une commande valide rattachée à un magasin et à un produit connus.
# Agrégation. Pour chaque magasin, SOMME de revenue_eur (à gauche) et NOMBRE de commandes (à droite).

# Nom, étiquette et couleur de chaque magasin, réutilisés dans les figures suivantes
STORE_NAME = dict(zip(store_summary['store_id'], store_summary['store_name'].fillna(store_summary['store_id'])))
STORE_LABEL = {s: f'{STORE_NAME[s]} ({s})' for s in store_summary['store_id']}
assert len(store_summary) <= len(SERIES_COLORS), 'more stores than distinct colours, use small multiples'
# La couleur suit le magasin (ordre des identifiants), pas son rang
STORE_COLOR = dict(zip(sorted(store_summary['store_id']), SERIES_COLORS))

# Barres horizontales, le plus gros chiffre d'affaires en haut
f1 = store_summary.iloc[::-1]
ylabels = [f'{STORE_NAME[s]}\n{s}, {c}, {fmt}' for s, c, fmt in
           zip(f1['store_id'], f1['city'].fillna('city unknown'), f1['store_format'].fillna('format unknown'))]
y = np.arange(len(f1))

# Deux graphiques côte à côte qui partagent les mêmes magasins (pas de double axe)
fig1, (ax_rev, ax_n) = plt.subplots(1, 2, figsize=(11, 1.0 + 0.75 * len(f1)), sharey=True,
                                    gridspec_kw={'width_ratios': [3, 2], 'wspace': 0.08})
ax_rev.barh(y, f1['total_revenue_eur'], height=0.55, color=BAR_COLOR)
ax_n.barh(y, f1['order_count'], height=0.55, color=CONTEXT_COLOR)
# Valeur écrite au bout de chaque barre (montant et part du total)
for yi, (v, share) in enumerate(zip(f1['total_revenue_eur'], f1['revenue_share_pct'])):
    ax_rev.text(v, yi, f'  {eur(v)}  ({share:.0f}%)', va='center', ha='left', color=INK_2, fontsize=9)
for yi, n in enumerate(f1['order_count']):
    ax_n.text(n, yi, f'  {n}', va='center', ha='left', color=INK_2, fontsize=9)

# Titres, axes et grille
ax_rev.set_yticks(y)
ax_rev.set_yticklabels(ylabels)
ax_rev.set_title('Total revenue (EUR, excl. VAT) and share of the total', loc='left')
ax_n.set_title('Number of orders', loc='left')
ax_rev.xaxis.set_major_formatter(eur_axis(f1['total_revenue_eur'].max()))
ax_rev.set_xlim(0, f1['total_revenue_eur'].max() * 1.32)
ax_n.set_xlim(0, f1['order_count'].max() * 1.18)
for ax in (ax_rev, ax_n):
    ax.grid(axis='x')
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', length=0)
add_titles(fig1, 'Figure 1. Which store records the most revenue, and from how many orders?',
           POPULATION, top_inches=1.05)
plt.show()

**Your interpretation**

State the question, unit and aggregation; then interpret the chart.

*Write your answer here.*

In [ ]:
# Figure 2. Distribution du chiffre d'affaires par commande, magasin par magasin
# Question posée. Combien vaut une commande typique dans chaque magasin, et à quel point les montants varient-ils ?
# Unité d'observation. Une commande valide (pas d'agrégation, chaque commande est un point).
# Lecture. La boîte contient la moitié centrale des commandes (de Q1 à Q3), le trait noir est la médiane,
# le losange est la moyenne et les ronds sont les commandes atypiques (au delà de 1,5 fois l'écart interquartile).
# Les commandes atypiques sont gardées, rien ne montre que ce sont des erreurs.
# Échelle. Les montants vont de quelques euros à plusieurs milliers. Sur un axe normal les grosses commandes
# écrasent toutes les boîtes, donc l'axe est logarithmique (chaque graduation multiplie le montant).
import textwrap
from matplotlib.ticker import NullFormatter

FIG2_LOG_SCALE = True
# Une liste de montants par magasin, dans le même ordre que la figure 1
groups = [analysis.loc[analysis['store_id'].eq(s), 'revenue_eur'].to_numpy() for s in store_order]

fig2, ax = plt.subplots(figsize=(11, 5.8))
ax.boxplot(groups, patch_artist=True, widths=0.5, showmeans=True,
           boxprops=dict(facecolor='#cde2fb', edgecolor=BAR_COLOR, linewidth=1.2),
           medianprops=dict(color=INK, linewidth=2),
           meanprops=dict(marker='D', markersize=6, markerfacecolor='#eb6834', markeredgecolor='white'),
           whiskerprops=dict(color=INK_2, linewidth=1), capprops=dict(color=INK_2, linewidth=1),
           flierprops=dict(marker='o', markersize=4, markerfacecolor='none',
                           markeredgecolor=INK_MUTED, alpha=0.9))
# Sous chaque boîte, le nom du magasin et le nombre de commandes (n)
ax.set_xticks(np.arange(1, len(store_order) + 1))
ax.set_xticklabels(['\n'.join(textwrap.wrap(STORE_NAME[s], 18)) + f'\n{s}, n = {len(g)} orders'
                    for s, g in zip(store_order, groups)])
# Axe vertical en échelle logarithmique avec des graduations lisibles en euros
lo, hi = analysis['revenue_eur'].min(), analysis['revenue_eur'].max()
if FIG2_LOG_SCALE:
    ax.set_yscale('log')
    ticks = [t for t in [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000]
             if lo / 1.5 <= t <= hi * 1.5]
    ax.set_yticks(ticks)
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_ylim(lo / 1.4, hi * 1.4)
    ax.set_ylabel('Revenue per order (EUR, excl. VAT), log scale')
else:
    ax.set_ylim(bottom=0)
    ax.set_ylabel('Revenue per order (EUR, excl. VAT)')
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: eur(v)))
ax.grid(axis='y')
ax.set_axisbelow(True)
ax.tick_params(axis='x', length=0)

# Légende qui explique chaque élément de la boîte
legend_items = [Patch(facecolor='#cde2fb', edgecolor=BAR_COLOR, label='Middle 50% of orders (Q1 to Q3)'),
                Line2D([0], [0], color=INK, linewidth=2, label='Median order'),
                Line2D([0], [0], marker='D', linestyle='none', markersize=6, markerfacecolor='#eb6834',
                       markeredgecolor='white', label='Mean order'),
                Line2D([0], [0], marker='o', linestyle='none', markersize=4, markerfacecolor='none',
                       markeredgecolor=INK_MUTED, label='Atypical order (beyond 1.5 x IQR), kept')]
ax.legend(handles=legend_items, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=9)
add_titles(fig2, 'Figure 2. How much is a typical order worth in each store?',
           POPULATION + ', stores in the same order as Figure 1')
plt.show()

# Les mêmes informations en tableau (nombre, moyenne, quartiles, minimum et maximum)
print('Order revenue by store in EUR')
display(analysis.groupby('store_id')['revenue_eur'].describe().loc[store_order]
        .rename(columns={'count': 'orders', '50%': 'median'}))

**Your interpretation**

State the question and observation unit; describe spread and a limitation.

*Write your answer here.*

In [ ]:
# Figure 3. Chiffre d'affaires jour par jour dans chaque magasin
# Question posée. Comment le chiffre d'affaires enregistré évolue-t-il chaque jour, et quels jours un magasin est-il absent ?
# Unité d'observation. Un couple magasin et jour du calendrier commun (1er au 28 avril).
# Agrégation. SOMME de revenue_eur par magasin et par date. Un jour sans commande reste vide (un trou dans la ligne),
# jamais 0, car l'extrait n'est pas complet et une absence ne prouve ni zéro vente ni fermeture.
# Mise en page. Un petit graphique par magasin, avec le même calendrier et la même échelle partout,
# car cinq lignes qui se croisent sur un seul graphique sont illisibles. Chaque magasin garde sa couleur.
from matplotlib.ticker import MaxNLocator

panel_stores = sorted(daily_revenue.columns)
weekend = [d for d in CALENDAR if d.dayofweek >= 5]
# Même échelle verticale pour tous les magasins, et repérage du jour le plus élevé
y_top = np.nanmax(daily_revenue.to_numpy()) * 1.18
peak_store = daily_revenue.max().idxmax()
peak_day = daily_revenue[peak_store].idxmax()

fig3, axes = plt.subplots(len(panel_stores), 1, figsize=(12, 1.3 * len(panel_stores) + 1.9),
                          sharex=True, sharey=True, gridspec_kw={'hspace': 0.6})
axes = np.atleast_1d(axes)
for ax, s in zip(axes, panel_stores):
    # Bandes grises claires pour les samedis et dimanches
    for d in weekend:
        ax.axvspan(d - pd.Timedelta(hours=12), d + pd.Timedelta(hours=12), color='#f0efec', linewidth=0, zorder=0)
    # La ligne s'interrompt toute seule les jours vides (NaN), les points montrent les jours isolés
    series = daily_revenue[s]
    ax.plot(series.index, series.to_numpy(), color=STORE_COLOR[s], linewidth=1.8, marker='o', markersize=4.5,
            markeredgecolor='white', markeredgewidth=0.8, zorder=3)
    # Étiquette du magasin avec sa couleur et le nombre de jours présents dans l'extrait
    key = Line2D([0], [0], color=STORE_COLOR[s], linewidth=1.8, marker='o', markersize=4.5,
                 markeredgecolor='white', markeredgewidth=0.8)
    ax.legend([key], [f'{STORE_LABEL[s]}, {int(series.notna().sum())} of {len(CALENDAR)} days with recorded orders'],
              loc='lower left', bbox_to_anchor=(0, 0.98), borderaxespad=0, handlelength=1.8, fontsize=9.5)
    # On écrit seulement la valeur du jour record pour expliquer le pic
    if s == peak_store:
        ax.annotate(f'{eur(series[peak_day])} on {peak_day:%d %b}', (peak_day, series[peak_day]),
                    xytext=(9, 0), textcoords='offset points', va='center', fontsize=9, color=INK_2)
    ax.set_ylim(0, y_top)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    ax.yaxis.set_major_formatter(eur_axis(y_top))
    ax.grid(axis='y')
    ax.set_axisbelow(True)

# Axe des dates commun en bas (une graduation tous les deux jours)
axes[-1].set_xlim(STUDY_START - pd.Timedelta(days=0.6), STUDY_END + pd.Timedelta(days=0.6))
axes[-1].xaxis.set_major_locator(mdates.DayLocator(interval=2))
axes[-1].xaxis.set_minor_locator(mdates.DayLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%a\n%d %b'))
axes[-1].set_xlabel('Order date (2025), grey bands are weekends')
fig3.text(0.065, 0.45, 'Recorded revenue per day (EUR, excl. VAT)', rotation=90, ha='center', va='center',
          color=INK_2, fontsize=10)
add_titles(fig3, 'Figure 3. How does recorded revenue evolve day by day in each store?',
           'One point is one store-day with at least one recorded order. A gap means no record in the extract, '
           f'not zero. Same calendar and scale in every panel, {N_ORDERS} orders, 1 to 28 April 2025.',
           top_inches=1.15)
plt.show()

**Your interpretation**

What does one point represent? How did you treat missing dates?

*Write your answer here.*

## 6. Write a decision brief
Write **six to eight sentences** addressed to the manager. Include:
- a clear answer to the original claim;
- at least two numerical results, with units and population;
- a distinction between total revenue and mean or median order revenue;
- a data-quality or integration limitation and a sample-coverage limitation;
- one plausible explanation labelled as a hypothesis;
- one proportionate next action and the extra data needed.

Avoid “Store X is best” without defining best. Do not claim causality, profitability or complete activity from this extract.

**Your interpretation**

Write your final decision brief here.

*Write your answer here.*

## 7. Export and reproduce
Save `store_summary.csv`, `category_summary.csv`, `reconciliation.csv`, `sales_review.csv` and `integration_review.csv` into `case_study_outputs/`. Save the three figures as PNG files. Use `index=False` for tables.

Before submitting:
- restart the kernel and run all cells from top to bottom;
- check that joins did not multiply rows and revenue totals reconcile;
- check that raw files remain unchanged;
- remove abandoned trials and explain all final decisions;
- make sure every member can explain one cleaning decision, one join control and every chart.

The deliverable is the entire completed notebook, including interpretations, not just the first figure.

In [ ]:
# Section 7. Export des résultats et vérifications de reproductibilité
# 7.1 Tableaux. On arrondit à 2 décimales seulement pour la présentation (montants et pourcentages).
money = lambda df: df.round({c: 2 for c in df.columns if c.endswith('_eur') or c.endswith('_pct')
                             or c in ('units_per_order',)})
exports = {
    'store_summary.csv': money(store_summary),
    'category_summary.csv': money(category_summary),
    'reconciliation.csv': money(reconciliation),
    'sales_review.csv': sales_review,
    'integration_review.csv': money(integration_review.assign(date=integration_review['date'].dt.strftime('%Y-%m-%d'))),
    # Fichiers en plus de ceux demandés, utiles comme preuves
    'daily_store_revenue.csv': money(daily_store_revenue.assign(date=daily_store_revenue['date'].dt.strftime('%Y-%m-%d'))),
    'reference_review.csv': pd.concat([stores_review.assign(table='stores'),
                                       products_review.assign(table='products')], ignore_index=True),
}
# index=False pour ne pas écrire la colonne d'index de pandas dans les CSV
for name, table in exports.items():
    table.to_csv(OUT_DIR / name, index=False)

# 7.2 Les trois figures en PNG
figures = {'fig1_total_revenue_by_store.png': fig1,
           'fig2_order_revenue_distribution.png': fig2,
           'fig3_daily_revenue_by_store.png': fig3}
for name, fig in figures.items():
    fig.savefig(OUT_DIR / name, dpi=150, bbox_inches='tight', facecolor='white')

# 7.3 Vérifications finales
# Les fichiers bruts sur le disque n'ont pas changé (même empreinte qu'au début)
assert {f: hashlib.sha256((DATA_DIR / f).read_bytes()).hexdigest() for f in RAW_FILES} == RAW_FINGERPRINTS, \
    'a raw file changed on disk'
# Les tableaux bruts en mémoire n'ont pas changé et sont identiques aux fichiers relus
for name, frame in [('sales', sales_raw), ('stores', stores_raw), ('products', products_raw)]:
    assert frame.equals(RAW_SNAPSHOTS[name]), f'raw object {name} was modified'
    assert frame.equals(pd.read_csv(DATA_DIR / f'case_{name}.csv', dtype='string')), f'{name} differs from disk'
# Les jointures n'ont pas multiplié les lignes et les totaux se réconcilient
assert len(orders_full) == len(sales_valid)
assert np.isclose(store_summary['total_revenue_eur'].sum(), analysis['revenue_eur'].sum())
assert len(sales_raw) == N_EXACT_DUP + len(sales_review) + len(analysis) + len(integration_review)

print('All final checks passed. Raw files unchanged (SHA-256), raw objects unchanged, '
      'joins kept row counts, totals reconcile.\n')
# Liste des fichiers produits, avec leur nombre de lignes et leur taille
written = pd.DataFrame([{'file': p.name, 'rows': str(len(exports[p.name])) if p.name in exports else '',
                         'size_kB': round(p.stat().st_size / 1024, 1)}
                        for p in sorted(OUT_DIR.iterdir()) if p.suffix in ('.csv', '.png')])
print(f'Files in {OUT_DIR}')
display(written)

### OPTIONAL / IF TIME ALLOWS — Investigate product mix
Create a store × category table of revenue or order counts. Use a labelled heatmap or compare category shares within each store. State the denominator explicitly. Does the mix help explain different order amounts? Treat this as descriptive evidence, not a causal proof.

In [ ]:
# Partie optionnelle. Mix produit, part de chaque catégorie dans le chiffre d'affaires de chaque magasin
# Unité d'observation. Une commande valide analysée.
# Agrégation. SOMME de revenue_eur par magasin et par catégorie, divisée par le total du magasin
# (le dénominateur est le chiffre d'affaires du magasin dans l'extrait, chaque ligne fait 100 %).
from matplotlib.colors import LinearSegmentedColormap

# Dégradé d'une seule couleur, du plus clair (petite part) au plus foncé (grande part)
blues = LinearSegmentedColormap.from_list('one_hue_blue', ['#f4f8fd', '#cde2fb', '#86b6ef', '#3987e5',
                                                           '#256abf', '#184f95', '#0d366b'])
heat = mix_share.loc[store_order]
vmax = float(np.ceil(heat.to_numpy().max() / 10) * 10)

fig4, ax = plt.subplots(figsize=(1.35 * heat.shape[1] + 3.2, 0.62 * heat.shape[0] + 1.9))
im = ax.imshow(heat.to_numpy(), cmap=blues, vmin=0, vmax=vmax, aspect='auto')
# Le pourcentage est écrit dans chaque case (en blanc sur les cases foncées)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.iat[i, j]
        ax.text(j, i, f'{v:.0f}%', ha='center', va='center', fontsize=9,
                color='white' if v > 0.55 * vmax else INK)
ax.set_xticks(np.arange(heat.shape[1]))
ax.set_xticklabels(heat.columns, rotation=0)
ax.set_yticks(np.arange(heat.shape[0]))
ax.set_yticklabels([STORE_LABEL[s] for s in heat.index])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig4.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("% of the store's revenue", color=INK_2)
cbar.outline.set_visible(False)
add_titles(fig4, "Optional. What share of each store's revenue comes from each category?",
           "Denominator is the store's total revenue in the analysed extract. Each row sums to 100%. "
           'Descriptive, not causal.', top_inches=0.85)
plt.show()

# Même calcul avec le nombre de commandes au lieu du chiffre d'affaires (les produits chers pèsent moins)
order_mix = (analysis.pivot_table(index='store_id', columns='category_label', values='order_id',
                                  aggfunc='count', fill_value=0).reindex(store_order))
order_mix = (100 * order_mix.div(order_mix.sum(axis=1), axis=0))[heat.columns]
print("Share of each store's orders by category in % (denominator is the store's order count)")
display(order_mix.round(1))

# Sauvegarde de la figure et du tableau optionnels
fig4.savefig(OUT_DIR / 'fig4_product_mix_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
(mix_share.loc[store_order].round(2).reset_index()
 .to_csv(OUT_DIR / 'store_category_revenue_share.csv', index=False))